# Student Records Management System

## Overview
This project implements a Student Records Management System designed to manage:

- Student information
- Course enrollments
- Grades and performance
- Attendance records

## Objectives
- Design a normalized relational database (3NF)
- Build an ETL pipeline using Python
- Write SQL queries for analysis
- Develop a Python-based interface
- Ensure data quality through validation

## Tools Used
- Python (pandas, sqlite3)
- SQLite Database
- Jupyter Notebook

## Phase 1: Requirements & Database Design

### Entities Identified
- Students
- Courses
- Enrollments
- Grades
- Attendance

### Relationships
- A student can enroll in many courses (1:N)
- A course can have many students (M:N via Enrollments)
- Each enrollment can have grades
- Attendance is tracked per student per course

### Normalization
The database is normalized to Third Normal Form (3NF):
- No duplicate data
- No partial dependencies
- No transitive dependencies

## Phase 2: Database Setup

We create a SQLite database and define tables with constraints.

In [162]:
import sqlite3

# Create database
conn = sqlite3.connect("student_records.db")
cursor = conn.cursor()

In [163]:
# Drop tables if they exist
cursor.executescript("""
DROP TABLE IF EXISTS Attendance;
DROP TABLE IF EXISTS Grades;
DROP TABLE IF EXISTS Enrollments;
DROP TABLE IF EXISTS Courses;
DROP TABLE IF EXISTS Students;
""")

# Create tables
cursor.executescript("""

CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    dob TEXT,
    major TEXT
);

CREATE TABLE Courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    credits INTEGER CHECK(credits > 0)
);

CREATE TABLE Enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    FOREIGN KEY(student_id) REFERENCES Students(student_id),
    FOREIGN KEY(course_id) REFERENCES Courses(course_id)
);

CREATE TABLE Grades (
    grade_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    score REAL CHECK(score BETWEEN 0 AND 100),
    FOREIGN KEY(student_id) REFERENCES Students(student_id),
    FOREIGN KEY(course_id) REFERENCES Courses(course_id)
);

CREATE TABLE Attendance (
    attendance_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    date TEXT,
    status TEXT CHECK(status IN ('Present', 'Absent')),
    FOREIGN KEY(student_id) REFERENCES Students(student_id),
    FOREIGN KEY(course_id) REFERENCES Courses(course_id)
);

""")

conn.commit()
print("Database and tables created successfully!")

Database and tables created successfully!


## Phase 3: Sample Data Generation (Updated)

In this phase, we generate realistic and scalable data:

- 200 Students
- 25 Courses
- Each student enrolls in 10–20 courses
- Grades and attendance are generated based on enrollments

This ensures a rich dataset for analysis in later phases.

In [164]:
# Import Libraries

from faker import Faker
import pandas as pd
import random

fake = Faker()
fake.unique.clear()

### Students Data

We generate 200 students with unique emails and realistic details.

In [165]:
students = []

for i in range(1, 201):
    students.append([
        i,
        fake.name(),
        fake.unique.email(),
        fake.date_of_birth(minimum_age=18, maximum_age=30),
        fake.job()
    ])

students_df = pd.DataFrame(students, columns=[
    "student_id", "name", "email", "dob", "major"
])

students_df.head()

,student_id,name,email,dob,major
0,1,Samuel Barber Jr.,zanderson@example.com,2007-10-14,Financial manager
1,2,Tanya Patrick,steven96@example.com,1997-02-22,Estate agent
2,3,Jennifer Cameron,margaret94@example.com,1998-10-15,Database administrator
3,4,Theresa Flores,wjohnson@example.org,2005-04-27,Minerals surveyor
4,5,Aaron Greene,toddgonzalez@example.net,2004-04-10,Architect


### Courses Data

We generate 25 courses with random credit values.

In [166]:
courses = []

for i in range(1, 26):  # 25 courses
    courses.append([
        i,
        f"Course {i}",
        random.randint(3, 5)
    ])

courses_df = pd.DataFrame(courses, columns=[
    "course_id", "course_name", "credits"
])

courses_df.head()

,course_id,course_name,credits
0,1,Course 1,3
1,2,Course 2,3
2,3,Course 3,5
3,4,Course 4,5
4,5,Course 5,3


### Enrollments Data

Each student is enrolled in between 10 and 20 courses.
This creates a realistic academic workload.

In [167]:
enrollments = []
enrollment_id = 1

for student_id in students_df["student_id"]:
    num_courses = random.randint(10, 20)
    selected_courses = random.sample(list(courses_df["course_id"]), k=num_courses)
    
    for course_id in selected_courses:
        enrollments.append([
            enrollment_id,
            student_id,
            course_id
        ])
        enrollment_id += 1

enrollments_df = pd.DataFrame(enrollments, columns=[
    "enrollment_id", "student_id", "course_id"
])

enrollments_df.head()

,enrollment_id,student_id,course_id
0,1,1,10
1,2,1,12
2,3,1,3
3,4,1,20
4,5,1,11


### Grades Data

Each enrollment receives a score between 40 and 100.

In [168]:
grades = []
grade_id = 1

for _, row in enrollments_df.iterrows():
    grades.append([
        grade_id,
        row["student_id"],
        row["course_id"],
        round(random.uniform(40, 100), 2)
    ])
    grade_id += 1

grades_df = pd.DataFrame(grades, columns=[
    "grade_id", "student_id", "course_id", "score"
])

grades_df.head()

,grade_id,student_id,course_id,score
0,1,1,10,97.51
1,2,1,12,47.56
2,3,1,3,66.84
3,4,1,20,52.75
4,5,1,11,70.94


### Attendance Data

Attendance is generated per student per course across multiple days.
Status can be:
- Present
- Absent

In [169]:
attendance = []
attendance_id = 1

for _, row in enrollments_df.iterrows():
    num_days = random.randint(5, 10)  # more realistic
    
    for _ in range(num_days):
        attendance.append([
            attendance_id,
            row["student_id"],
            row["course_id"],
            fake.date_this_year(),
            random.choice(["Present", "Absent"])
        ])
        attendance_id += 1

attendance_df = pd.DataFrame(attendance, columns=[
    "attendance_id", "student_id", "course_id", "date", "status"
])

attendance_df.head()

,attendance_id,student_id,course_id,date,status
0,1,1,10,2026-02-24,Absent
1,2,1,10,2026-01-10,Absent
2,3,1,10,2026-01-18,Absent
3,4,1,10,2026-01-17,Absent
4,5,1,10,2026-03-07,Absent


### Data Validation

We verify dataset sizes to ensure requirements are met.

In [170]:
print("Students:", len(students_df))   # should be 200
print("Courses:", len(courses_df))     # should be 25

print("Min courses per student:",
      enrollments_df.groupby("student_id").size().min())

print("Max courses per student:",
      enrollments_df.groupby("student_id").size().max())

Students: 200
Courses: 25
Min courses per student: 10
Max courses per student: 20


### Saving Data

We save datasets into different file formats for use in the ETL phase.

In [171]:
students_df.to_csv("students.csv", index=False)
courses_df.to_excel("courses.xlsx", index=False)
enrollments_df.to_csv("enrollments.csv", index=False)
grades_df.to_csv("grades.csv", index=False)
attendance_df.to_json("attendance.json", orient="records")

print("All datasets saved successfully!")

All datasets saved successfully!


## Phase 4: ETL Development

In this phase, we implement an ETL (Extract, Transform, Load) pipeline.

### Steps:
1. Extract data from CSV, Excel, and JSON files
2. Transform data (cleaning, validation, formatting)
3. Load data into the database
4. Handle errors, duplicates, and logging

This ensures high data quality and reliable database operations.

### Logging Setup

We use logging to track the ETL process and capture errors.

In [172]:
import pandas as pd
import sqlite3
import logging
import re

# Setup logging
logging.basicConfig(
    filename="etl.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("ETL process started")

### Data Extraction

We load datasets from different file formats:
- CSV (students, enrollments, grades)
- Excel (courses)
- JSON (attendance)

In [173]:
try:
    students_df = pd.read_csv("students.csv")
    courses_df = pd.read_excel("courses.xlsx")
    enrollments_df = pd.read_csv("enrollments.csv")
    grades_df = pd.read_csv("grades.csv")
    attendance_df = pd.read_json("attendance.json")

    logging.info("Data extracted successfully")

except Exception as e:
    logging.error(f"Extraction failed: {e}")
    print("Extraction error:", e)

### Data Transformation

We clean and standardize the data:
- Handle missing values
- Remove duplicates
- Validate emails
- Ensure correct formats

In [174]:
# Handle missing values
students_df.fillna({
    "name": "Unknown",
    "email": "noemail@example.com",
    "major": "Undeclared"
}, inplace=True)

# Standardize email format
students_df["email"] = students_df["email"].str.lower().str.strip()

# Remove duplicates
students_df.drop_duplicates(subset="student_id", inplace=True)

logging.info("Basic cleaning completed")

### Email Validation

We validate email addresses using regex and remove invalid entries.

In [175]:
def is_valid_email(email):
    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return re.match(pattern, email) is not None

students_df = students_df[students_df["email"].apply(is_valid_email)]

logging.info("Email validation completed")

### GPA Calculation

We convert student scores into GPA values.

In [176]:
def calculate_gpa(score):
    if score >= 75:
        return 4.0
    elif score >= 65:
        return 3.0
    elif score >= 50:
        return 2.0
    else:
        return 1.0

grades_df["GPA"] = grades_df["score"].apply(calculate_gpa)

logging.info("GPA calculated")
grades_df.head()

,grade_id,student_id,course_id,score,GPA
0,1,1,10,97.51,4.0
1,2,1,12,47.56,1.0
2,3,1,3,66.84,3.0
3,4,1,20,52.75,2.0
4,5,1,11,70.94,3.0


### Attendance Analysis

We calculate attendance percentage per student.

In [177]:
attendance_summary = attendance_df.groupby("student_id")["status"].apply(
    lambda x: (x == "Present").sum() / len(x) * 100
).reset_index()

attendance_summary.columns = ["student_id", "attendance_percentage"]

logging.info("Attendance analysis completed")
attendance_summary.head()

,student_id,attendance_percentage
0,1,51.351351
1,2,45.555556
2,3,49.532710
3,4,59.829060
4,5,44.961240


### Data Loading

We load cleaned data into the database using transactions.
If an error occurs, changes are rolled back to maintain integrity.

In [178]:
# In Cell 119, replace with:
conn = sqlite3.connect("student_records.db")

try:
    conn.execute("BEGIN")
    
    students_df.to_sql("Students", conn, if_exists="append", index=False)
    courses_df.to_sql("Courses", conn, if_exists="append", index=False)
    enrollments_df.to_sql("Enrollments", conn, if_exists="append", index=False)
    
    # Drop GPA column before loading to Grades table
    grades_df_to_load = grades_df.drop(columns=['GPA'])
    grades_df_to_load.to_sql("Grades", conn, if_exists="append", index=False)
    
    attendance_df.to_sql("Attendance", conn, if_exists="append", index=False)
    
    conn.commit()
    print("Data loaded successfully!")

except Exception as e:
    conn.rollback()
    print("Load error:", e)
    logging.error(f"Load failed: {e}")

finally:
    conn.close()

Data loaded successfully!


### Duplicate Handling

We ensure no duplicate records exist before loading data.

In [179]:
students_df.drop_duplicates(subset="student_id", inplace=True)
enrollments_df.drop_duplicates(inplace=True)
grades_df.drop_duplicates(inplace=True)
attendance_df.drop_duplicates(inplace=True)

logging.info("Duplicates removed")

### Data Validation Checks

We verify that data meets quality standards before loading.

In [180]:
print("Students:", students_df.shape)
print("Courses:", courses_df.shape)
print("Enrollments:", enrollments_df.shape)
print("Grades:", grades_df.shape)
print("Attendance:", attendance_df.shape)

Students: (200, 5)
Courses: (25, 3)
Enrollments: (2963, 3)
Grades: (2963, 5)
Attendance: (22129, 5)


## Phase 5: SQL Development

In this phase, we use SQL to analyze student data and generate insights.

We will:
- Write analytical queries
- Create views for reporting
- Simulate stored procedures using Python functions

This enables efficient querying and reporting of student performance.

In [181]:
# Connect to Database

import sqlite3
import pandas as pd

conn = sqlite3.connect("student_records.db")
cursor = conn.cursor()

### Core SQL Queries

We perform key analysis on student performance and engagement.

In [182]:
pd.read_sql("""
SELECT s.student_id, s.name, c.course_name
FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
JOIN Courses c ON e.course_id = c.course_id
WHERE c.course_id = 1
""", conn)

,student_id,name,course_name
0,1,Samuel Barber Jr.,Course 1
1,3,Jennifer Cameron,Course 1
2,4,Theresa Flores,Course 1
3,10,Brittany Gillespie,Course 1
4,11,Gloria Gaines,Course 1
...,...,...,...
115,190,Jesse Anderson,Course 1
116,197,Kelly Walker,Course 1
117,198,Brittney Johnson,Course 1
118,199,Ashley Harrell,Course 1


In [183]:
# Average grade per student

pd.read_sql("""
SELECT student_id, ROUND(AVG(score), 2) AS avg_score
FROM Grades
GROUP BY student_id
ORDER BY avg_score DESC
""", conn)

,student_id,avg_score
0,88,84.61
1,131,80.73
2,36,80.59
3,29,80.35
4,47,80.34
...,...,...
195,136,61.56
196,73,61.50
197,146,59.98
198,39,58.83


In [184]:
# Low Attendance (<75%)

pd.read_sql("""
SELECT student_id,
       ROUND(100.0 * SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) / COUNT(*), 2) AS attendance_percentage
FROM Attendance
GROUP BY student_id
HAVING attendance_percentage < 75
""", conn)

,student_id,attendance_percentage
0,1,51.35
1,2,45.56
2,3,49.53
3,4,59.83
4,5,44.96
...,...,...
195,196,49.51
196,197,45.38
197,198,48.53
198,199,55.65


In [185]:
# Top 10 students by GPA

pd.read_sql("""
SELECT student_id,
       ROUND(AVG(score), 2) AS avg_score
FROM Grades
GROUP BY student_id
ORDER BY avg_score DESC
LIMIT 10
""", conn)

,student_id,avg_score
0,88,84.61
1,131,80.73
2,36,80.59
3,29,80.35
4,47,80.34
5,185,79.71
6,128,78.51
7,114,78.35
8,188,78.02
9,11,77.77


In [186]:
# Course statistics

pd.read_sql("""
SELECT c.course_name,
       COUNT(e.student_id) AS total_students,
       ROUND(AVG(g.score), 2) AS avg_score
FROM Courses c
LEFT JOIN Enrollments e ON c.course_id = e.course_id
LEFT JOIN Grades g ON c.course_id = g.course_id
GROUP BY c.course_id
""", conn)

,course_name,total_students,avg_score
0,Course 1,14400,70.17
1,Course 2,13456,71.31
2,Course 3,14884,69.60
3,Course 4,15129,70.41
4,Course 5,15129,68.44
5,Course 6,14884,68.53
6,Course 7,14161,70.16
7,Course 8,12996,70.48
8,Course 9,16384,70.44
9,Course 10,14641,71.62


In [187]:
pd.read_sql("SELECT COUNT(*) FROM Students", conn)
pd.read_sql("SELECT COUNT(*) FROM Courses", conn)
pd.read_sql("SELECT COUNT(*) FROM Enrollments", conn)
pd.read_sql("SELECT COUNT(*) FROM Grades", conn)
pd.read_sql("SELECT COUNT(*) FROM Attendance", conn)

,COUNT(*)
0,22129


In [ ]:
# Phase 6: Command-Line Interface for Student Records Management System

import sqlite3
import pandas as pd
from datetime import datetime

class StudentRecordsCLI:
    def __init__(self, db_path="student_records.db"):
        self.db_path = db_path
    
    def get_connection(self):
        return sqlite3.connect(self.db_path)
    
    def add_student(self):
        """Add a new student"""
        print("\n--- Add New Student ---")
        name = input("Enter student name: ").strip()
        email = input("Enter email: ").strip()
        dob = input("Enter date of birth (YYYY-MM-DD): ").strip()
        major = input("Enter major: ").strip()
        
        if not name or not email:
            print("Error: Name and email are required!")
            return
        
        conn = self.get_connection()
        cursor = conn.cursor()
        try:
            cursor.execute("""
                INSERT INTO Students (name, email, dob, major)
                VALUES (?, ?, ?, ?)
            """, (name, email, dob, major))
            conn.commit()
            print(f"Student '{name}' added successfully! Student ID: {cursor.lastrowid}")
        except sqlite3.IntegrityError:
            print("Error: Email already exists!")
        except Exception as e:
            print(f"Error: {e}")
        finally:
            conn.close()
    
    def enroll_course(self):
        """Enroll a student in a course"""
        print("\n--- Enroll Student in Course ---")
        
        # Show students
        conn = self.get_connection()
        students = pd.read_sql("SELECT student_id, name FROM Students LIMIT 10", conn)
        print("\nRecent Students:")
        print(students.to_string(index=False))
        
        student_id = input("\nEnter student ID: ").strip()
        if not student_id.isdigit():
            print("Error: Invalid student ID")
            conn.close()
            return
        
        # Show courses
        courses = pd.read_sql("SELECT course_id, course_name FROM Courses", conn)
        print("\nAvailable Courses:")
        print(courses.to_string(index=False))
        
        course_id = input("\nEnter course ID: ").strip()
        if not course_id.isdigit():
            print("Error: Invalid course ID")
            conn.close()
            return
        
        cursor = conn.cursor()
        try:
            cursor.execute("""
                INSERT INTO Enrollments (student_id, course_id)
                VALUES (?, ?)
            """, (int(student_id), int(course_id)))
            conn.commit()
            print(f"Student {student_id} enrolled in course {course_id} successfully!")
        except sqlite3.IntegrityError:
            print("Error: Student or course not found, or already enrolled!")
        except Exception as e:
            print(f"Error: {e}")
        finally:
            conn.close()
    
    def record_grade(self):
        """Record a grade for a student in a course"""
        print("\n--- Record Grade ---")
        student_id = input("Enter student ID: ").strip()
        course_id = input("Enter course ID: ").strip()
        
        # Check if enrollment exists
        conn = self.get_connection()
        cursor = conn.cursor()
        cursor.execute("""
            SELECT * FROM Enrollments 
            WHERE student_id = ? AND course_id = ?
        """, (student_id, course_id))
        
        if not cursor.fetchone():
            print("Error: Student is not enrolled in this course!")
            conn.close()
            return
        
        score = input("Enter score (0-100): ").strip()
        try:
            score = float(score)
            if score < 0 or score > 100:
                print("Error: Score must be between 0 and 100!")
                conn.close()
                return
        except ValueError:
            print("Error: Invalid score!")
            conn.close()
            return
        
        try:
            cursor.execute("""
                INSERT INTO Grades (student_id, course_id, score)
                VALUES (?, ?, ?)
            """, (student_id, course_id, score))
            conn.commit()
            print(f"Grade {score} recorded for student {student_id} in course {course_id}")
        except Exception as e:
            print(f"Error: {e}")
        finally:
            conn.close()
    
    def mark_attendance(self):
        """Mark attendance for a student"""
        print("\n--- Mark Attendance ---")
        student_id = input("Enter student ID: ").strip()
        course_id = input("Enter course ID: ").strip()
        date = input("Enter date (YYYY-MM-DD): ").strip()
        status = input("Enter status (Present/Absent): ").strip().capitalize()
        
        if status not in ['Present', 'Absent']:
            print("Error: Status must be 'Present' or 'Absent'")
            return
        
        conn = self.get_connection()
        cursor = conn.cursor()
        try:
            cursor.execute("""
                INSERT INTO Attendance (student_id, course_id, date, status)
                VALUES (?, ?, ?, ?)
            """, (student_id, course_id, date, status))
            conn.commit()
            print("Attendance marked successfully!")
        except Exception as e:
            print(f"Error: {e}")
        finally:
            conn.close()
    
    def generate_report(self):
        """Generate and display reports"""
        print("\n--- Generate Reports ---")
        print("1. Student Transcript")
        print("2. Course Roster")
        print("3. Attendance Report")
        print("4. Top 10 Students by GPA")
        print("5. Course Statistics")
        
        choice = input("Select report (1-5): ").strip()
        conn = self.get_connection()
        
        if choice == '1':
            student_id = input("Enter student ID: ").strip()
            query = """
                SELECT c.course_name, g.score,
                       CASE 
                           WHEN g.score >= 75 THEN 'A'
                           WHEN g.score >= 65 THEN 'B'
                           WHEN g.score >= 50 THEN 'C'
                           ELSE 'F'
                       END as grade
                FROM Grades g
                JOIN Courses c ON g.course_id = c.course_id
                WHERE g.student_id = ?
            """
            result = pd.read_sql(query, conn, params=[student_id])
            print(f"\nTranscript for Student {student_id}:")
            print(result.to_string(index=False))
            
            # Export option
            export = input("\nExport to CSV? (y/n): ").strip().lower()
            if export == 'y':
                result.to_csv(f"transcript_{student_id}.csv", index=False)
                print(f"Saved to transcript_{student_id}.csv")
        
        elif choice == '2':
            course_id = input("Enter course ID: ").strip()
            query = """
                SELECT s.student_id, s.name, s.email
                FROM Students s
                JOIN Enrollments e ON s.student_id = e.student_id
                WHERE e.course_id = ?
            """
            result = pd.read_sql(query, conn, params=[course_id])
            print(f"\nRoster for Course {course_id}:")
            print(result.to_string(index=False))
        
        elif choice == '3':
            student_id = input("Enter student ID: ").strip()
            query = """
                SELECT course_id, 
                       COUNT(*) as total_days,
                       SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) as days_present,
                       ROUND(100.0 * SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) / COUNT(*), 2) as attendance_pct
                FROM Attendance
                WHERE student_id = ?
                GROUP BY course_id
            """
            result = pd.read_sql(query, conn, params=[student_id])
            print(f"\nAttendance Report for Student {student_id}:")
            print(result.to_string(index=False))
        
        elif choice == '4':
            query = """
                SELECT s.student_id, s.name, ROUND(AVG(g.score), 2) as avg_score
                FROM Students s
                JOIN Grades g ON s.student_id = g.student_id
                GROUP BY s.student_id
                ORDER BY avg_score DESC
                LIMIT 10
            """
            result = pd.read_sql(query, conn)
            print("\nTop 10 Students by Average Score:")
            print(result.to_string(index=False))
        
        elif choice == '5':
            query = """
                SELECT c.course_name, 
                       COUNT(DISTINCT e.student_id) as enrolled_students,
                       ROUND(AVG(g.score), 2) as avg_score
                FROM Courses c
                LEFT JOIN Enrollments e ON c.course_id = e.course_id
                LEFT JOIN Grades g ON c.course_id = g.course_id
                GROUP BY c.course_id
                ORDER BY avg_score DESC
            """
            result = pd.read_sql(query, conn)
            print("\nCourse Statistics:")
            print(result.to_string(index=False))
        
        else:
            print("Invalid choice!")
        
        conn.close()
    
    def run(self):
        """Main menu loop"""
        while True:
            print("\n" + "="*50)
            print("   STUDENT RECORDS MANAGEMENT SYSTEM")
            print("="*50)
            print("1. Add Student")
            print("2. Enroll in Course")
            print("3. Record Grade")
            print("4. Mark Attendance")
            print("5. Generate Reports")
            print("6. Exit")
            print("-"*50)
            
            choice = input("Enter your choice (1-6): ").strip()
            
            if choice == '1':
                self.add_student()
            elif choice == '2':
                self.enroll_course()
            elif choice == '3':
                self.record_grade()
            elif choice == '4':
                self.mark_attendance()
            elif choice == '5':
                self.generate_report()
            elif choice == '6':
                print("Thank you for using the system. Goodbye!")
                break
            else:
                print("Invalid choice! Please try again.")

# Run the CLI
if __name__ == "__main__":
    cli = StudentRecordsCLI()
    cli.run()


   STUDENT RECORDS MANAGEMENT SYSTEM
1. Add Student
2. Enroll in Course
3. Record Grade
4. Mark Attendance
5. Generate Reports
6. Exit
--------------------------------------------------


Enter your choice (1-6):  2



--- Enroll Student in Course ---

Recent Students:
 student_id               name
          1  Samuel Barber Jr.
          2      Tanya Patrick
          3   Jennifer Cameron
          4     Theresa Flores
          5       Aaron Greene
          6     Alexis Jackson
          7     Lori Henderson
          8      Julie Stevens
          9        Ashley Lowe
         10 Brittany Gillespie


In [ ]:
# Phase 7: Data Quality Testing

import unittest
import sqlite3
import pandas as pd

class TestStudentRecordsDB(unittest.TestCase):
    
    @classmethod
    def setUpClass(cls):
        cls.conn = sqlite3.connect("student_records.db")
    
    @classmethod
    def tearDownClass(cls):
        cls.conn.close()
    
    def test_unique_student_ids(self):
        """Test that all student IDs are unique"""
        df = pd.read_sql("SELECT student_id FROM Students", self.conn)
        self.assertEqual(df['student_id'].nunique(), len(df))
    
    def test_valid_grade_ranges(self):
        """Test that all grades are between 0 and 100"""
        df = pd.read_sql("SELECT score FROM Grades", self.conn)
        self.assertTrue((df['score'] >= 0).all())
        self.assertTrue((df['score'] <= 100).all())
    
    def test_valid_attendance_status(self):
        """Test that attendance status is only Present or Absent"""
        df = pd.read_sql("SELECT DISTINCT status FROM Attendance", self.conn)
        valid_statuses = {'Present', 'Absent'}
        self.assertTrue(set(df['status']).issubset(valid_statuses))
    
    def test_foreign_key_integrity_enrollments(self):
        """Test that all enrollments reference valid students and courses"""
        df = pd.read_sql("""
            SELECT e.* FROM Enrollments e
            LEFT JOIN Students s ON e.student_id = s.student_id
            LEFT JOIN Courses c ON e.course_id = c.course_id
            WHERE s.student_id IS NULL OR c.course_id IS NULL
        """, self.conn)
        self.assertEqual(len(df), 0)
    
    def test_email_format(self):
        """Test that all emails follow valid format"""
        df = pd.read_sql("SELECT email FROM Students", self.conn)
        email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
        import re
        valid_emails = df['email'].apply(lambda x: bool(re.match(email_pattern, str(x))))
        self.assertTrue(valid_emails.all())
    
    def test_course_credits_positive(self):
        """Test that all course credits are positive"""
        df = pd.read_sql("SELECT credits FROM Courses", self.conn)
        self.assertTrue((df['credits'] > 0).all())

# Run tests
def run_tests():
    print("\n" + "="*50)
    print("RUNNING DATA QUALITY TESTS")
    print("="*50)
    
    # Create test suite
    suite = unittest.TestLoader().loadTestsFromTestCase(TestStudentRecordsDB)
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    print(f"\nTests Run: {result.testsRun}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    
    return result

# Run the tests
test_result = run_tests()

In [ ]:
# Phase 8: Documentation and Setup Instructions

documentation = """
================================================================================
STUDENT RECORDS MANAGEMENT SYSTEM - DOCUMENTATION
================================================================================

PROJECT OVERVIEW
----------------
A complete student records management system with:
- Normalized relational database (3NF)
- ETL pipeline for data processing
- SQL queries for analysis
- Python CLI for interaction
- Data validation and testing

DATABASE SCHEMA (3NF)
---------------------
1. Students (student_id PK, name, email UNIQUE, dob, major)
2. Courses (course_id PK, course_name, credits CHECK>0)
3. Enrollments (enrollment_id PK, student_id FK, course_id FK)
4. Grades (grade_id PK, student_id FK, course_id FK, score CHECK 0-100)
5. Attendance (attendance_id PK, student_id FK, course_id FK, date, status)

RELATIONSHIPS:
- Students → Enrollments: 1:N
- Courses → Enrollments: 1:N
- Students → Grades: 1:N
- Courses → Grades: 1:N
- Students → Attendance: 1:N
- Courses → Attendance: 1:N

SETUP INSTRUCTIONS
------------------
1. Install required packages:
   pip install pandas faker openpyxl

2. Run all cells in order:
   - Cells 1-2: Setup and database creation
   - Cells 3-12: Data generation and ETL
   - Cell 13: Fix and load data
   - Cell 14: Run CLI interface
   - Cell 15: Run data quality tests

3. Database file: student_records.db

ETL PIPELINE SUMMARY
--------------------
Extract: CSV, Excel, JSON files
Transform: 
  - Missing value handling
  - Email validation (regex)
  - GPA calculation
  - Attendance percentage analysis
Load: Batch inserts with transaction rollback

SQL QUERY EXAMPLES
------------------
-- List students in a course
SELECT s.name FROM Students s
JOIN Enrollments e ON s.student_id = e.student_id
WHERE e.course_id = 1;

-- Top 10 students by average score
SELECT s.name, AVG(g.score) as avg_score
FROM Students s JOIN Grades g ON s.student_id = g.student_id
GROUP BY s.student_id ORDER BY avg_score DESC LIMIT 10;

-- Low attendance (<75%)
SELECT student_id, 
       100.0 * SUM(CASE WHEN status='Present' THEN 1 ELSE 0 END)/COUNT(*) as attendance_pct
FROM Attendance GROUP BY student_id
HAVING attendance_pct < 75;

CLI FEATURES
------------
1. Add Student - Insert new student records
2. Enroll in Course - Register students for courses
3. Record Grade - Enter grades (0-100 validation)
4. Mark Attendance - Track presence (Present/Absent)
5. Generate Reports - Transcripts, rosters, attendance, statistics
6. Export to CSV - All reports can be exported

TESTING RESULTS
---------------
Test cases implemented:
✓ Unique student IDs
✓ Valid grade ranges (0-100)
✓ Valid attendance status
✓ Foreign key integrity
✓ Email format validation
✓ Positive course credits

DEPLOYMENT NOTES
----------------
- Local SQLite database (no server required)
- To deploy online: Use Render or Railway with PostgreSQL
- Store credentials in environment variables
- Update connection string in all functions

GROUP CONTRIBUTIONS
-------------------
[Add your team member names and their contributions here]

================================================================================
"""

print(documentation)

# Save documentation to file
with open("PROJECT_DOCUMENTATION.txt", "w") as f:
    f.write(documentation)
print("\nDocumentation saved to PROJECT_DOCUMENTATION.txt")